# Práctica: Clustering de jugadores en FIFA22 con aprendizaje no supervisado

### Objetivos
En esta práctica exploraremos distintas técnicas de **aprendizaje no supervisado** aplicadas al análisis de datos de jugadores de FIFA22.

El propósito es **agrupar jugadores según sus habilidades** y contrastar con posiciones reales en el campo, siguiendo la línea del artículo *"Clustering in Game Analysis on FIFA22 Official Players Data"* (IEEE AiDAS, 2022) y tomando como inspiración un trabajo destacado del curso 2024–25.


- Preprocesamiento y selección de atributos.
- Reducción de dimensionalidad con **PCA**.
- Comparación de **K-Means**, **Clustering Jerárquico (HC)** y **DBSCAN**.
- Exploración de **algoritmos alternativos** (GMM, Spectral, OPTICS, Birch...).
- Interpretación mediante **métricas** y **visualización**.


## 1. Preparación del entorno

**Tareas:**
1. Importar librerías necesarias (`pandas`, `numpy`, `matplotlib`, `seaborn`, `sklearn`).
2. Cargar el dataset oficial de FIFA22 desde Kaggle (`players_22.csv`).
3. Mostrar dimensiones y primeras filas.


In [47]:

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

df = pd.read_csv('players_22.csv')
df.head()

C:\Users\Usuario\AppData\Local\Temp\ipykernel_14992\1029159340.py:13: DtypeWarning: Columns (25,108) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('players_22.csv')


,sofifa_id,player_url,short_name,long_name,player_positions,overall,potential,value_eur,wage_eur,age,...,lcb,cb,rcb,rb,gk,player_face_url,club_logo_url,club_flag_url,nation_logo_url,nation_flag_url
0,158023,https://sofifa.com/player/158023/lionel-messi/...,L. Messi,Lionel Andrés Messi Cuccittini,"RW, ST, CF",93,93,78000000.0,320000.0,34,...,50+3,50+3,50+3,61+3,19+3,https://cdn.sofifa.net/players/158/023/22_120.png,https://cdn.sofifa.net/teams/73/60.png,https://cdn.sofifa.net/flags/fr.png,https://cdn.sofifa.net/teams/1369/60.png,https://cdn.sofifa.net/flags/ar.png
1,188545,https://sofifa.com/player/188545/robert-lewand...,R. Lewandowski,Robert Lewandowski,ST,92,92,119500000.0,270000.0,32,...,60+3,60+3,60+3,61+3,19+3,https://cdn.sofifa.net/players/188/545/22_120.png,https://cdn.sofifa.net/teams/21/60.png,https://cdn.sofifa.net/flags/de.png,https://cdn.sofifa.net/teams/1353/60.png,https://cdn.sofifa.net/flags/pl.png
2,20801,https://sofifa.com/player/20801/c-ronaldo-dos-...,Cristiano Ronaldo,Cristiano Ronaldo dos Santos Aveiro,"ST, LW",91,91,45000000.0,270000.0,36,...,53+3,53+3,53+3,60+3,20+3,https://cdn.sofifa.net/players/020/801/22_120.png,https://cdn.sofifa.net/teams/11/60.png,https://cdn.sofifa.net/flags/gb-eng.png,https://cdn.sofifa.net/teams/1354/60.png,https://cdn.sofifa.net/flags/pt.png
3,190871,https://sofifa.com/player/190871/neymar-da-sil...,Neymar Jr,Neymar da Silva Santos Júnior,"LW, CAM",91,91,129000000.0,270000.0,29,...,50+3,50+3,50+3,62+3,20+3,https://cdn.sofifa.net/players/190/871/22_120.png,https://cdn.sofifa.net/teams/73/60.png,https://cdn.sofifa.net/flags/fr.png,NaN,https://cdn.sofifa.net/flags/br.png
4,192985,https://sofifa.com/player/192985/kevin-de-bruy...,K. De Bruyne,Kevin De Bruyne,"CM, CAM",91,91,125500000.0,350000.0,30,...,69+3,69+3,69+3,75+3,21+3,https://cdn.sofifa.net/players/192/985/22_120.png,https://cdn.sofifa.net/teams/10/60.png,https://cdn.sofifa.net/flags/gb-eng.png,https://cdn.sofifa.net/teams/1325/60.png,https://cdn.sofifa.net/flags/be.png


## 2. Exploración y limpieza de datos

**Posibles tareas: (opcional)**
1. Elimina jugadores **porteros (GK)** y sus variables específicas.
2. Descarta columnas **no numéricas o irrelevantes** (identificadores, URL, nombres, club, nacionalidad, etc.).
3. Elimina atributos **posicionales derivados** (LS, ST, RS, LW, RW, ...) de los datos de entrada. Pueden servirte como etiquetas para comprobar si el clustering es efectivo.
4. Rellena valores nulos (por ejemplo, con la **media**).
5. Aplica un **filtro de calidad**: conserva jugadores con `overall` **> 70**.
6. Verifica el tamaño final del dataset y el % de nulos.


In [48]:
# Me cargo a los porteros porque no me interesan para el análisis, ya que sus estadísticas son muy diferentes al resto de jugadores y podrían distorsionar el análisis.
df_no_gk = df[~df['player_positions'].str.contains('GK', na=False)]
df_no_gk.head()

,sofifa_id,player_url,short_name,long_name,player_positions,overall,potential,value_eur,wage_eur,age,...,lcb,cb,rcb,rb,gk,player_face_url,club_logo_url,club_flag_url,nation_logo_url,nation_flag_url
0,158023,https://sofifa.com/player/158023/lionel-messi/...,L. Messi,Lionel Andrés Messi Cuccittini,"RW, ST, CF",93,93,78000000.0,320000.0,34,...,50+3,50+3,50+3,61+3,19+3,https://cdn.sofifa.net/players/158/023/22_120.png,https://cdn.sofifa.net/teams/73/60.png,https://cdn.sofifa.net/flags/fr.png,https://cdn.sofifa.net/teams/1369/60.png,https://cdn.sofifa.net/flags/ar.png
1,188545,https://sofifa.com/player/188545/robert-lewand...,R. Lewandowski,Robert Lewandowski,ST,92,92,119500000.0,270000.0,32,...,60+3,60+3,60+3,61+3,19+3,https://cdn.sofifa.net/players/188/545/22_120.png,https://cdn.sofifa.net/teams/21/60.png,https://cdn.sofifa.net/flags/de.png,https://cdn.sofifa.net/teams/1353/60.png,https://cdn.sofifa.net/flags/pl.png
2,20801,https://sofifa.com/player/20801/c-ronaldo-dos-...,Cristiano Ronaldo,Cristiano Ronaldo dos Santos Aveiro,"ST, LW",91,91,45000000.0,270000.0,36,...,53+3,53+3,53+3,60+3,20+3,https://cdn.sofifa.net/players/020/801/22_120.png,https://cdn.sofifa.net/teams/11/60.png,https://cdn.sofifa.net/flags/gb-eng.png,https://cdn.sofifa.net/teams/1354/60.png,https://cdn.sofifa.net/flags/pt.png
3,190871,https://sofifa.com/player/190871/neymar-da-sil...,Neymar Jr,Neymar da Silva Santos Júnior,"LW, CAM",91,91,129000000.0,270000.0,29,...,50+3,50+3,50+3,62+3,20+3,https://cdn.sofifa.net/players/190/871/22_120.png,https://cdn.sofifa.net/teams/73/60.png,https://cdn.sofifa.net/flags/fr.png,NaN,https://cdn.sofifa.net/flags/br.png
4,192985,https://sofifa.com/player/192985/kevin-de-bruy...,K. De Bruyne,Kevin De Bruyne,"CM, CAM",91,91,125500000.0,350000.0,30,...,69+3,69+3,69+3,75+3,21+3,https://cdn.sofifa.net/players/192/985/22_120.png,https://cdn.sofifa.net/teams/10/60.png,https://cdn.sofifa.net/flags/gb-eng.png,https://cdn.sofifa.net/teams/1325/60.png,https://cdn.sofifa.net/flags/be.png


In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19239 entries, 0 to 19238
Columns: 110 entries, sofifa_id to nation_flag_url
dtypes: float64(16), int64(44), object(50)
memory usage: 16.1+ MB


In [50]:
df_no_gk['preferred_foot_num'] = df_no_gk['preferred_foot'].map({'Left': 0, 'Right': 1}).fillna(1)
df_no_gk['real_face_num'] = df_no_gk['real_face'].map({False: 0, True: 1}).fillna(0)

C:\Users\Usuario\AppData\Local\Temp\ipykernel_14992\2530187100.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_no_gk['preferred_foot_num'] = df_no_gk['preferred_foot'].map({'Left': 0, 'Right': 1}).fillna(1)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_14992\2530187100.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_no_gk['real_face_num'] = df_no_gk['real_face'].map({False: 0, True: 1}).fillna(0)


In [51]:
work_map = {'Low/Low':0, 'Medium/Low':1, 'Low/Medium':1, 'Medium/Medium':2, 
            'High/Low':2, 'Medium/High':3, 'High/Medium':3, 'High/High':3}
df_no_gk['work_rate_num'] = df_no_gk['work_rate'].map(work_map).fillna(2)

C:\Users\Usuario\AppData\Local\Temp\ipykernel_14992\928505336.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_no_gk['work_rate_num'] = df_no_gk['work_rate'].map(work_map).fillna(2)


In [52]:
body_map = {'Lean':1, 'Normal':0, 'Stocky':3, 'Lean (170-)':1, 'Normal (170-)':0}
df_no_gk['body_type_num'] = df_no_gk['body_type'].map(body_map).fillna(0)

C:\Users\Usuario\AppData\Local\Temp\ipykernel_14992\1611921016.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_no_gk['body_type_num'] = df_no_gk['body_type'].map(body_map).fillna(0)


In [53]:
# Ahora me voy a cargar todas las columnas que creo que no sean necesarias, todas aquellas que no contentan variables numéricas o que no aporten información relevante para el análisis.
columns_to_drop = ['sofifa_id', 'player_url', 'short_name', 'player_position','long_name', 'dob', 'club_jersey_number', 'nation_jersey_number','release_clause_eur','club_position', 'club_joined', 'club_contract_valid_until', 'nation_position','club_jersey_number', 'club_loaned_from', 'nationality_name' , 'nationality_id','nation_team_id','league_name', 'club_name', 'club_team_id' ,'player_face_url', 'player_tags','player_traits','club_logo_url', 'club_flag_url', 'nation_logo_url', 'nation_flag_url']
object_cols = df_no_gk.select_dtypes(exclude=[np.number]).columns.tolist()
df_stats = df_no_gk.drop(columns=object_cols).dropna()
df_stats.head()

,sofifa_id,overall,potential,value_eur,wage_eur,age,height_cm,weight_kg,club_team_id,league_level,...,goalkeeping_diving,goalkeeping_handling,goalkeeping_kicking,goalkeeping_positioning,goalkeeping_reflexes,goalkeeping_speed,preferred_foot_num,real_face_num,work_rate_num,body_type_num


## 3. Selección de características y normalización

**Tareas guiadas:**
1. (Opcional) Selección de características para eliminar redundancia (p.ej. árbol de decisión por-feature con R² alto).
2. Escala los datos (StandardScaler o MinMaxScaler).
3. (Opcional) Aplica **transformación log** si fuera necesario.
4. Visualiza histogramas antes y después.


## 4. Reducción de dimensionalidad con PCA 

**Tareas guiadas:**
1. Ajusta **PCA** y representa la **varianza explicada acumulada**.
2. Elige componentes suficientes para ≥ 85–90% de varianza.
3. Visualiza los datos en las **dos primeras componentes**. 
4. Interpreta las **cargas (loadings)** de PCA.


In [54]:
# Pensar como puedo reducir el número de variables que no me hagan falta. Nos quedamos con un número de componentes concreto.
# Hay columnas que no tienen sentido. Un ejemplo sería el sueldo.


## 5. K-Means

**Tareas guiadas:**
1. Aplica **K-Means** sobre `X_pca` con `k ∈ {3,4,8,14}`.
2. Calcula **Inercia (SSE)** y **Silhouette** para cada k.
3. Dibuja **curva del codo** y **silhouette vs k**.
4. Visualiza los clusters en PC1–PC2 y **comenta** si son interpretables.


## 6. Clustering Jerárquico (HC)

**Tareas guiadas:**
1. Usa **AgglomerativeClustering** con `linkage`: `ward`, `complete`, `average`, `single`.
2. Métrica: Euclídea (Manhattan cuando sea compatible).
3. Representa **dendrograma** (con `scipy` + `linkage`).
4. Calcula **Silhouette** y compara con K-Means.


##  7. DBSCAN

**Tareas guiadas:**
1. Estima `eps` con el **k-distance plot** (`NearestNeighbors`).
2. Prueba varios `eps` y `min_samples`.
3. Cuenta nº de clústeres y **ruido**.
4. Calcula **Silhouette** (excluyendo ruido) y compara.


## 8. Comparativa de modelos

Completa y amplia con los resultados de tus experimentos:

| Modelo       | Configuración             | Nº clusters | Silhouette | Observaciones |
|:-------------|:--------------------------|------------:|-----------:|:--------------|
| K-Means      | PCA + k=3                 |             |            |               |
| Hierarchical | Ward                      |             |            |               |
| DBSCAN       | eps=?, min_samples=?      |             |            |               |

**Reflexiona:**
- ¿Qué modelo ofrece agrupamientos más interpretables?
- ¿Influye PCA en la calidad del clustering?
- ¿Qué atributos parecen definir mejor los roles?


## 9. Conclusiones

**Preguntas guía:**
- ¿Qué algoritmo separa mejor los grupos?
- ¿Cómo ayuda PCA a visualizar e interpretar?
- ¿Qué mejoras propondrías? (p. ej., usar otras métricas como **Davies–Bouldin**, o sub-clustering por rol).


## Referencias
- *Clustering in Game Analysis on FIFA22 Official Players Data*, IEEE AiDAS (2022).
- Scikit-learn: https://scikit-learn.org/stable/modules/clustering.html
- Kaggle: *FIFA Player Stats Database* (players_22.csv)
